# 1. Load data 

In [1]:
import pandas as pd

# Base dataset
mod = pd.read_parquet("datasets/raw/selected_mod_data_slim.parquet")

# Vendor datasets
v1 = pd.read_parquet("datasets/raw/vendor1_appends_combined.parquet")
v2 = pd.read_parquet("datasets/raw/vendor2_appended.parquet")
v3 = pd.read_parquet("datasets/raw/vendor3_appended.parquet")

In [2]:
print("MOD:", mod.shape)
print("V1 :", v1.shape)
print("V2 :", v2.shape)
print("V3 :", v3.shape)

MOD: (356448, 19)
V1 : (100000, 593)
V2 : (696906, 22)
V3 : (356448, 80)


# 2. Check merge keys

In [3]:
common_v1 = set(mod.columns) & set(v1.columns)
common_v2 = set(mod.columns) & set(v2.columns)
common_v3 = set(mod.columns) & set(v3.columns)

print("MOD & V1:", common_v1)
print("MOD & V2:", common_v2)
print("MOD & V3:", common_v3)

MOD & V1: {'record_id'}
MOD & V2: {'qpid', 'state', 'record_id', 'zip4'}
MOD & V3: {'state', 'record_id'}


# 3. Check uniquenss at intended keys

In [4]:
print("v1 duplicate record_id:", v1["record_id"].duplicated().sum())

print("v2 duplicate record_id + year:",
      v2[["record_id", "year"]].duplicated().sum())

print("v3 duplicate record_id + year:",
      v3[["record_id", "year"]].duplicated().sum())

v1 duplicate record_id: 0
v2 duplicate record_id + year: 0
v3 duplicate record_id + year: 0


# 4. Prepare vendor columns

In [5]:
v2_features = v2.drop(columns=["state", "zip5", "zip4"], errors="ignore")
v3_features = v3.drop(columns=["state", "zip5"], errors="ignore")

# 5. Merge

In [6]:
# Make merge key types consistent
mod["vendor_year"] = mod["vendor_year"].astype("int64")

v2_features["year"] = v2_features["year"].astype("int64")
v3_features["year"] = v3_features["year"].astype("int64")

In [7]:
df = (
    mod
    .merge(
        v1,
        on="record_id",
        how="left",
        suffixes=("", "_v1")
    )
    .merge(
        v2_features,
        left_on=["record_id", "vendor_year"],
        right_on=["record_id", "year"],
        how="left",
        suffixes=("", "_v2")
    )
    .merge(
        v3_features,
        left_on=["record_id", "vendor_year"],
        right_on=["record_id", "year"],
        how="left",
        suffixes=("", "_v3")
    )
)

print("mod shape:", mod.shape)
print("merged shape:", df.shape)

mod shape: (356448, 19)
merged shape: (356448, 706)


# 6. Sanity check after merge

In [8]:
print("Duplicate original rows after merge:",
      df[["record_id", "mod140_year"]].duplicated().sum())

print("Vendor 1 coverage:", df.filter(like="individual_id").notna().any(axis=1).mean())
print("Vendor 2 coverage:", df["year"].notna().mean())
print("Vendor 3 coverage:", df["year_v3"].notna().mean() if "year_v3" in df.columns else "check v3 year column")

Duplicate original rows after merge: 4835
Vendor 1 coverage: 0.947094667384864
Vendor 2 coverage: 0.9966671155399947
Vendor 3 coverage: 0.7228936619086094


In [9]:
grain_cols = [
    "record_id",
    "cur_term_eff_dt",
    "cur_term_xptn_dt",
    "mod140_year"
]

print("Original mod duplicate grain:")
print(mod[grain_cols].duplicated().sum())

print("Merged duplicate grain:")
print(df[grain_cols].duplicated().sum())

Original mod duplicate grain:
0
Merged duplicate grain:
0


In [10]:
# Save merged dataset
df.to_csv("datasets/merged.csv", index=False)
# Save parquet version (recommended)
df.to_parquet("datasets/merged.parquet", index=False)

In [11]:
# Columns to remove
drop_cols = [
    # ID columns
    #"record_id",
    "individual_id",
    "household_id",
    "qpid",

    # Sensitive / protected class related columns
    "percent_asian",
    "percent_black",
    "percent_hispanic",
    "percent_white",
    "ethnicity_group_v2",
    "race_v2",
    "hispanic_country_code"
]

# Drop columns safely
df_clean = df.drop(columns=drop_cols, errors="ignore")

# Quick shape check
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

# Save cleaned dataset
df_clean.to_csv("datasets/merged_clean.csv", index=False)

# Save parquet version (recommended)
df_clean.to_parquet("datasets/merged_clean.parquet", index=False)

print("Cleaned datasets saved successfully.")

Original shape: (356448, 706)
Cleaned shape: (356448, 696)
Cleaned datasets saved successfully.
